In [ ]:
import os
import sys
import pyspark
from pyspark.sql import SparkSession

print("Python:", sys.version)
print("PySpark:", pyspark.__version__)
print("JAVA_HOME antes:", os.environ.get("JAVA_HOME"))
print("SPARK_HOME antes:", os.environ.get("SPARK_HOME"))
print("HADOOP_HOME antes:", os.environ.get("HADOOP_HOME"))

print("JAVA_HOME depois:", os.environ.get("JAVA_HOME"))
print("PATH tem java?:", r"\bin" in os.environ["PATH"])

spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("test")
    .getOrCreate()
)


Python: 3.12.6 (tags/v3.12.6:a4a2d2b, Sep  6 2024, 20:11:23) [MSC v.1940 64 bit (AMD64)]
PySpark: 4.1.1
JAVA_HOME antes: C:\Program Files\Java\jdk-22
SPARK_HOME antes: C:\spark
HADOOP_HOME antes: C:\hadoop
JAVA_HOME depois: C:\Program Files\Java\jdk-22
PATH tem java?: True


In [2]:
print("Caminho raiz do notebook:", os.getcwd())

Caminho raiz do notebook: d:\Programacao\DESAFIOS\VERITY


In [3]:
caminho_base = os.path.join(os.getcwd(), "data", "output")
print("Caminho base:", caminho_base)
print(os.path.exists(caminho_base))

try:
    df = spark.read.parquet(caminho_base)
    df.show()
except Exception as e:
    print(type(e))
    print(e)

Caminho base: d:\Programacao\DESAFIOS\VERITY\data\output
True
+------+-----------+--------+-------------------+--------+---------------+---------+-------------+
|amount|customer_id|event_id|        ingested_at|order_id|    source_file|   status|business_date|
+------+-----------+--------+-------------------+--------+---------------+---------+-------------+
| 210.0|        599|      19|2026-04-13 08:00:00|    1999|     batch_late|     paid|   2026-04-11|
| 120.0|        601|       6|2026-04-11 05:10:00|    2001|batch_001_retry|     paid|   2026-04-11|
|  80.0|        602|       7|2026-04-11 05:12:00|    2002|   batch_update|     paid|   2026-04-11|
| 200.0|        603|       3|2026-04-11 05:02:00|    2003|      batch_001|     paid|   2026-04-11|
|  50.0|        604|       4|2026-04-11 05:03:00|    2004|      batch_001|cancelled|   2026-04-11|
| 300.0|        605|       5|2026-04-11 05:04:00|    2005|      batch_001|     paid|   2026-04-11|
| 101.0|        620|      20|2026-04-11 08:00:0

In [4]:
df.printSchema()
df.show(20, truncate=False)
print(f"Total de linhas no output: {df.count()}")

root
 |-- amount: double (nullable = true)
 |-- customer_id: long (nullable = true)
 |-- event_id: long (nullable = true)
 |-- ingested_at: timestamp (nullable = true)
 |-- order_id: long (nullable = true)
 |-- source_file: string (nullable = true)
 |-- status: string (nullable = true)
 |-- business_date: date (nullable = true)

+------+-----------+--------+-------------------+--------+---------------+---------+-------------+
|amount|customer_id|event_id|ingested_at        |order_id|source_file    |status   |business_date|
+------+-----------+--------+-------------------+--------+---------------+---------+-------------+
|210.0 |599        |19      |2026-04-13 08:00:00|1999    |batch_late     |paid     |2026-04-11   |
|120.0 |601        |6       |2026-04-11 05:10:00|2001    |batch_001_retry|paid     |2026-04-11   |
|80.0  |602        |7       |2026-04-11 05:12:00|2002    |batch_update   |paid     |2026-04-11   |
|200.0 |603        |3       |2026-04-11 05:02:00|2003    |batch_001      |p

In [5]:
from pyspark.sql import functions as F

df.groupBy("business_date") \
  .count() \
  .orderBy("business_date") \
  .show()

+-------------+-----+
|business_date|count|
+-------------+-----+
|   2026-04-11|   20|
|   2026-04-12|   19|
|   2026-04-13|   19|
+-------------+-----+



In [6]:
df.groupBy("business_date").agg(
    F.count("*").alias("qtd_registros"),
    F.sum("amount").alias("valor_total"),
    F.avg("amount").alias("valor_medio")
).orderBy("business_date").show(truncate=False)

+-------------+-------------+-----------+------------------+
|business_date|qtd_registros|valor_total|valor_medio       |
+-------------+-------------+-----------+------------------+
|2026-04-11   |20           |2607.0     |130.35            |
|2026-04-12   |19           |2586.0     |136.10526315789474|
|2026-04-13   |19           |2560.0     |134.73684210526315|
+-------------+-------------+-----------+------------------+



In [7]:
df.groupBy("status") \
  .count() \
  .orderBy(F.desc("count")) \
  .show()

+---------+-----+
|   status|count|
+---------+-----+
|     paid|   36|
|cancelled|   12|
|  pending|   10|
+---------+-----+



In [8]:
df.groupBy("business_date", "status") \
  .count() \
  .orderBy("business_date", "status") \
  .show()

+-------------+---------+-----+
|business_date|   status|count|
+-------------+---------+-----+
|   2026-04-11|cancelled|    4|
|   2026-04-11|     paid|   13|
|   2026-04-11|  pending|    3|
|   2026-04-12|cancelled|    4|
|   2026-04-12|     paid|   12|
|   2026-04-12|  pending|    3|
|   2026-04-13|cancelled|    4|
|   2026-04-13|     paid|   11|
|   2026-04-13|  pending|    4|
+-------------+---------+-----+



In [9]:
df.groupBy("order_id") \
  .count() \
  .filter(F.col("count") > 1) \
  .show()

+--------+-----+
|order_id|count|
+--------+-----+
+--------+-----+



In [10]:
print("Total de order_id distintos:", df.select("order_id").distinct().count())
print("Total de linhas:", df.count())

Total de order_id distintos: 58
Total de linhas: 58


In [11]:
for c in df.columns:
    nulls = df.filter(F.col(c).isNull()).count()
    print(f"{c}: {nulls} nulos")

amount: 0 nulos
customer_id: 0 nulos
event_id: 0 nulos
ingested_at: 0 nulos
order_id: 0 nulos
source_file: 0 nulos
status: 0 nulos
business_date: 0 nulos


In [12]:
df.orderBy(F.desc("amount")).show(10, truncate=False)

+------+-----------+--------+-------------------+--------+------------+---------+-------------+
|amount|customer_id|event_id|ingested_at        |order_id|source_file |status   |business_date|
+------+-----------+--------+-------------------+--------+------------+---------+-------------+
|500.0 |613        |16      |2026-04-13 07:02:00|2013    |batch_003   |paid     |2026-04-13   |
|400.0 |609        |11      |2026-04-12 06:03:00|2009    |batch_002   |paid     |2026-04-12   |
|300.0 |605        |5       |2026-04-11 05:04:00|2005    |batch_001   |paid     |2026-04-11   |
|220.0 |607        |13      |2026-04-12 07:00:00|2007    |batch_update|paid     |2026-04-12   |
|210.0 |599        |19      |2026-04-13 08:00:00|1999    |batch_late  |paid     |2026-04-11   |
|200.0 |603        |3       |2026-04-11 05:02:00|2003    |batch_001   |paid     |2026-04-11   |
|142.0 |661        |61      |2026-04-13 11:00:00|2061    |batch_auto  |paid     |2026-04-13   |
|141.0 |660        |60      |2026-04-12 

In [13]:
df.select(
    F.min("business_date").alias("data_min"),
    F.max("business_date").alias("data_max")
).show()

+----------+----------+
|  data_min|  data_max|
+----------+----------+
|2026-04-11|2026-04-13|
+----------+----------+



In [14]:
from pyspark.sql import functions as F

print("1. Total de registros finais")
print(df.count())

print("2. Intervalo de datas processado")
df.select(
    F.min("business_date").alias("min_date"),
    F.max("business_date").alias("max_date")
).show()

print("3. Registros por data")
df.groupBy("business_date").count().orderBy("business_date").show()

print("4. Distribuição por status")
df.groupBy("status").count().orderBy(F.desc("count")).show()

print("5. Soma financeira por data")
df.groupBy("business_date").agg(
    F.sum("amount").alias("total_amount"),
    F.avg("amount").alias("avg_amount")
).orderBy("business_date").show()

print("6. Verificação de duplicidade por order_id")
df.groupBy("order_id").count().filter("count > 1").show()

1. Total de registros finais
58
2. Intervalo de datas processado
+----------+----------+
|  min_date|  max_date|
+----------+----------+
|2026-04-11|2026-04-13|
+----------+----------+

3. Registros por data
+-------------+-----+
|business_date|count|
+-------------+-----+
|   2026-04-11|   20|
|   2026-04-12|   19|
|   2026-04-13|   19|
+-------------+-----+

4. Distribuição por status
+---------+-----+
|   status|count|
+---------+-----+
|     paid|   36|
|cancelled|   12|
|  pending|   10|
+---------+-----+

5. Soma financeira por data
+-------------+------------+------------------+
|business_date|total_amount|        avg_amount|
+-------------+------------+------------------+
|   2026-04-11|      2607.0|            130.35|
|   2026-04-12|      2586.0|136.10526315789474|
|   2026-04-13|      2560.0|134.73684210526315|
+-------------+------------+------------------+

6. Verificação de duplicidade por order_id
+--------+-----+
|order_id|count|
+--------+-----+
+--------+-----+

